# 00 — Setup (Kaggle)

Cấu hình môi trường cho **Kaggle Notebooks**.

## Kaggle Storage Model

| Đường dẫn | Mục đích | Persist? |
|-----------|---------|----------|
| `/kaggle/input/<slug>/` | Dataset read-only | ✅ Luôn có |
| `/kaggle/working/` | Output — Save Version để giữ | ✅ Sau Save Version |

## Cần chuẩn bị trên Kaggle trước khi chạy

**Dataset 1**: `genai-vimd` — Upload thư mục `ViMD/` từ Drive lên Kaggle Dataset
- Cấu trúc: `ViMD/data/*.parquet`
- Slug: `genai-vimd` → available tại `/kaggle/input/genai-vimd/`

**Dataset 2** (optional): `disa-model-artifacts` — checkpoints từ run trước
- Slug: `disa-model-artifacts` → `/kaggle/input/disa-model-artifacts/`

**Notebook settings**:
- Accelerator: **GPU T4 x2** hoặc **GPU P100**
- Internet: **ON**

**Thứ tự chạy**: 00 → 01 → 03 → 04 → 05 → 06 → 07

In [ ]:
# ── Cell 0: Detect Environment + Configure Paths ──────────────────────────────
import os, sys
from pathlib import Path

# ─── Environment detection ────────────────────────────────────────────────────
IS_KAGGLE = os.path.exists('/kaggle/working')
IS_COLAB  = 'google.colab' in sys.modules or os.path.exists('/content/drive')
IS_LOCAL  = not IS_KAGGLE and not IS_COLAB

env_name = 'Kaggle' if IS_KAGGLE else ('Colab' if IS_COLAB else 'Local')
print(f'Running on: {env_name}')

# ─── Kaggle Paths ────────────────────────────────────────────────────────────
if IS_KAGGLE:
    # /kaggle/input/<slug>/ — read-only dataset mount
    # Thay 'genai-vimd' bằng slug thực tế của dataset bạn tạo
    KAGGLE_INPUT   = '/kaggle/input'
    VIMD_SLUG      = 'genai_vimd_dataset'          # ← sửa nếu slug khác
    ARTIFACT_SLUG  = 'disa-model-artifacts' # ← slug chứa checkpoints cũ (optional)

    VIMD_PATH      = os.path.join(KAGGLE_INPUT, VIMD_SLUG, 'ViMD', 'data')
    PREV_CKPT_DIR  = os.path.join(KAGGLE_INPUT, ARTIFACT_SLUG)  # checkpoints cũ

    # /kaggle/working/ — writable, persist khi Save Version
    PROJECT_ROOT   = '/kaggle/working'
    PROCESSED_PATH = '/kaggle/working/storage/processed'
    DATASET_EXPORT = '/kaggle/working/storage/datasets/hf_dataset'
    WEIGHTS_DIR    = '/kaggle/working/storage/weights'
    CHECKPOINT_DIR = '/kaggle/working/storage/checkpoints'
    LOGS_DIR       = '/kaggle/working/storage/logs'
    EXPORTS_DIR    = '/kaggle/working/storage/exports'

    # Clone code từ GitHub vào working dir (nếu chưa có)
    REPO_URL       = 'https://github.com/justeg4n/DisaTravel_AI.git'  # ← sửa URL
    CODE_DIR       = '/kaggle/working/project'

# ─── Colab Paths (fallback) ───────────────────────────────────────────────────
elif IS_COLAB:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
    DRIVE_ROOT     = '/content/drive/MyDrive/GenAI_ViMD'
    PROJECT_ROOT   = DRIVE_ROOT
    VIMD_PATH      = os.path.join(DRIVE_ROOT, 'ViMD', 'data')
    PROCESSED_PATH = os.path.join(DRIVE_ROOT, 'storage', 'processed')
    DATASET_EXPORT = os.path.join(DRIVE_ROOT, 'storage', 'datasets', 'hf_dataset')
    WEIGHTS_DIR    = os.path.join(DRIVE_ROOT, 'storage', 'weights')
    CHECKPOINT_DIR = os.path.join(DRIVE_ROOT, 'storage', 'checkpoints')
    LOGS_DIR       = os.path.join(DRIVE_ROOT, 'storage', 'logs')
    EXPORTS_DIR    = os.path.join(DRIVE_ROOT, 'storage', 'exports')
    CODE_DIR       = DRIVE_ROOT
    PREV_CKPT_DIR  = CHECKPOINT_DIR

# ─── Local Paths (fallback) ───────────────────────────────────────────────────
else:
    _nb = Path(os.path.abspath('.')).resolve()
    PROJECT_ROOT   = str(_nb.parent.parent) if _nb.name == 'notebooks' else str(_nb)
    VIMD_PATH      = os.path.join(PROJECT_ROOT, 'ViMD', 'data')
    PROCESSED_PATH = os.path.join(PROJECT_ROOT, 'storage', 'processed')
    DATASET_EXPORT = os.path.join(PROJECT_ROOT, 'storage', 'datasets', 'hf_dataset')
    WEIGHTS_DIR    = os.path.join(PROJECT_ROOT, 'storage', 'weights')
    CHECKPOINT_DIR = os.path.join(PROJECT_ROOT, 'storage', 'checkpoints')
    LOGS_DIR       = os.path.join(PROJECT_ROOT, 'storage', 'logs')
    EXPORTS_DIR    = os.path.join(PROJECT_ROOT, 'storage', 'exports')
    CODE_DIR       = PROJECT_ROOT
    PREV_CKPT_DIR  = CHECKPOINT_DIR

# ─── Export env vars ──────────────────────────────────────────────────────────
for k, v in [
    ('IS_KAGGLE',      str(IS_KAGGLE)),
    ('IS_COLAB',       str(IS_COLAB)),
    ('PROJECT_ROOT',   PROJECT_ROOT),
    ('VIMD_PATH',      VIMD_PATH),
    ('PROCESSED_PATH', PROCESSED_PATH),
    ('DATASET_EXPORT', DATASET_EXPORT),
    ('WEIGHTS_DIR',    WEIGHTS_DIR),
    ('CHECKPOINT_DIR', CHECKPOINT_DIR),
    ('LOGS_DIR',       LOGS_DIR),
    ('EXPORTS_DIR',    EXPORTS_DIR),
    ('CODE_DIR',       CODE_DIR),
]:
    os.environ[k] = v

print(f'PROJECT_ROOT   : {PROJECT_ROOT}')
print(f'VIMD_PATH      : {VIMD_PATH}')
print(f'DATASET_EXPORT : {DATASET_EXPORT}')
print(f'CHECKPOINT_DIR : {CHECKPOINT_DIR}')

In [ ]:
# ── Cell 1: Clone code từ GitHub (Kaggle only) ────────────────────────────────
import subprocess

if IS_KAGGLE:
    if not os.path.exists(os.path.join(CODE_DIR, 'training')):
        print(f'Cloning {REPO_URL} → {CODE_DIR}...')
        r = subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, CODE_DIR],
                           capture_output=True, text=True)
        if r.returncode == 0:
            print(f'✓ Code cloned to {CODE_DIR}')
        else:
            print(f'⚠️  Clone error: {r.stderr[-200:]}')
            print('   Hãy set REPO_URL đúng trong Cell 0')
    else:
        print(f'✓ Code đã có tại {CODE_DIR}')

    if CODE_DIR not in sys.path:
        sys.path.insert(0, CODE_DIR)
    print(f'sys.path: {CODE_DIR}')
else:
    print('ℹ️  Không phải Kaggle — skip git clone')

In [ ]:
# ── Cell 2: Kiểm tra ViMD data + Artifact checkpoints ────────────────────────
import glob

# ViMD parquet
if os.path.isdir(VIMD_PATH):
    parquet_files = glob.glob(os.path.join(VIMD_PATH, '*.parquet'))
    parquet_files += glob.glob(os.path.join(VIMD_PATH, '**', '*.parquet'), recursive=True)
    parquet_files = sorted(set(parquet_files))
    print(f'✓ ViMD parquet: {len(parquet_files)} shards')
    if parquet_files:
        print(f'  Sample: {[os.path.basename(f) for f in parquet_files[:3]]}')
else:
    print(f'❌ VIMD_PATH không tồn tại: {VIMD_PATH}')
    if IS_KAGGLE:
        print(f'   Hãy add dataset "{VIMD_SLUG}" vào notebook này')
        print(f'   Kaggle → notebook settings → Data → Add Dataset')
        # Xem các dataset đang mount
        if os.path.isdir('/kaggle/input'):
            print(f'   Datasets đang mount: {os.listdir("/kaggle/input")}')

# Kiểm tra checkpoints cũ
if IS_KAGGLE and os.path.isdir(PREV_CKPT_DIR):
    print(f'\n✓ Previous artifacts: {PREV_CKPT_DIR}')
    print(f'  Contents: {os.listdir(PREV_CKPT_DIR)[:5]}')
else:
    print('\nℹ️  Không có artifact dataset cũ — bắt đầu từ đầu')

In [ ]:
# ── Cell 3: Tạo thư mục output ────────────────────────────────────────────────
dirs_to_create = [
    PROCESSED_PATH,
    os.path.join(PROCESSED_PATH, 'features'),
    DATASET_EXPORT,
    WEIGHTS_DIR,
    os.path.join(WEIGHTS_DIR, 'dialect'),
    os.path.join(WEIGHTS_DIR, 'asr_wav2vec2'),
    os.path.join(WEIGHTS_DIR, 'translation'),
    os.path.join(WEIGHTS_DIR, 'xtts-v2'),
    os.path.join(CHECKPOINT_DIR, 'asr_wav2vec2'),
    os.path.join(CHECKPOINT_DIR, 'translation'),
    os.path.join(CHECKPOINT_DIR, 'tts'),
    LOGS_DIR,
    os.path.join(LOGS_DIR, 'mlflow'),
    EXPORTS_DIR,
    os.path.join(EXPORTS_DIR, 'onnx'),
    os.path.join(EXPORTS_DIR, 'torchscript'),
]
for d in dirs_to_create:
    Path(d).mkdir(parents=True, exist_ok=True)

print('✓ Tất cả thư mục tạo xong')

if IS_KAGGLE:
    # Restore checkpoints từ artifact dataset nếu có
    import shutil
    for stage in ['asr_wav2vec2', 'dialect', 'translation', 'tts']:
        src = os.path.join(PREV_CKPT_DIR, 'checkpoints', stage)
        dst = os.path.join(CHECKPOINT_DIR, stage)
        if os.path.isdir(src) and not os.listdir(dst):
            shutil.copytree(src, dst, dirs_exist_ok=True)
            print(f'  ✓ Restored checkpoint: {stage}')
    for stage in ['asr_wav2vec2', 'dialect', 'translation', 'xtts-v2']:
        src = os.path.join(PREV_CKPT_DIR, 'weights', stage)
        dst = os.path.join(WEIGHTS_DIR, stage)
        if os.path.isdir(src) and not os.listdir(dst):
            shutil.copytree(src, dst, dirs_exist_ok=True)
            print(f'  ✓ Restored weights: {stage}')

In [ ]:
# ── Cell 4: Cài dependencies ──────────────────────────────────────────────────
# Kaggle đã có: torch, transformers, datasets, sklearn — chỉ cài thêm
import subprocess

def pip_install(*pkgs, quiet=True):
    q = ['-q'] if quiet else []
    r = subprocess.run([sys.executable, '-m', 'pip', 'install'] + q + list(pkgs),
                       capture_output=True, text=True)
    return r.returncode == 0

print('Cài extra dependencies...')
# Kaggle đã có: torch, transformers, numpy, pandas, scikit-learn
pip_install('evaluate>=0.4.0', 'jiwer>=3.0.0')
pip_install('librosa>=0.10.0', 'soundfile', 'noisereduce', 'audiomentations')
pip_install('accelerate>=0.28.0', 'optimum')
pip_install('mlflow>=2.10.0', 'sacrebleu', 'sentencepiece', 'psutil')
pip_install('faster-whisper')

ok_tts = pip_install('TTS')
print(f'  TTS: {"✓" if ok_tts else "⚠️  skip"}')

print('\n✓ Dependencies installed')

In [ ]:
# ── Cell 5: Kiểm tra GPU ──────────────────────────────────────────────────────
import torch

print('=' * 60)
print(f'Environment  : {env_name}')
print(f'PyTorch      : {torch.__version__}')
print(f'CUDA         : {torch.cuda.is_available()}')

if torch.cuda.is_available():
    n_gpu   = torch.cuda.device_count()
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU ({n_gpu}x)    : {gpu_name}')
    print(f'VRAM/GPU     : {vram_gb:.1f} GB')
    if IS_KAGGLE:
        print('Kaggle tips:')
        print('  30h GPU/week, session up to ~9h')
        print('  Save Version sau mỗi stage hoàn tất!')
        if 'T4' in gpu_name:
            print('  T4 → batch_size=8 (wav2vec2-base)')
        elif 'P100' in gpu_name:
            print('  P100 → batch_size=16 (wav2vec2-base)')
else:
    print('❌ Không có GPU!')
    print('  Kaggle: Settings → Accelerator → GPU T4 x2')

try:
    import psutil
    ram = psutil.virtual_memory()
    print(f'RAM          : {ram.total/1e9:.1f} GB | {ram.available/1e9:.1f} GB free')
except ImportError:
    pass

print('=' * 60)

In [ ]:
# ── Cell 6: Pipeline State ────────────────────────────────────────────────────
steps = [
    ('01_dataset_prep', os.path.join(DATASET_EXPORT, 'vietspeech-vimd')),
    ('03_asr_wav2vec2', os.path.join(WEIGHTS_DIR, 'asr_wav2vec2', 'config.json')),
    ('04_translation',  os.path.join(WEIGHTS_DIR, 'translation', 'config.json')),
    ('05_tts',          os.path.join(WEIGHTS_DIR, 'xtts-v2', 'config.json')),
    ('07_onnx_export',  os.path.join(EXPORTS_DIR, 'onnx')),
]

print('Pipeline State:')
for name, path in steps:
    done   = os.path.exists(path)
    status = '✅ Done' if done else '⏳ Pending'
    print(f'  {name:30s} {status}')

if IS_KAGGLE:
    print()
    print('⚠️  QUAN TRỌNG — Kaggle workflow:')
    print('  1. Chạy xong mỗi stage → nhấn "Save Version" trên Kaggle UI')
    print('  2. Output lưu vào /kaggle/working/ và commit thành version')
    print('  3. Để dùng lại ở session sau: add output làm input dataset')
    print()
    print('✓ Setup complete! Chạy 01_dataset_prep.ipynb tiếp theo.')